# Weight initialization (Xavier)

_Deep Learning_

**Start the weights at small, just-right random values. Zeros or huge values break training.**

Before training, every weight needs a starting value. The choice matters a lot.

     If all weights start at zero, every neuron computes the same thing and they never split apart. Learning is stuck.

---

This notebook is a practice scaffold for the **Weight initialization (Xavier)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Build the network

We stack three `nn.Linear` layers with ReLU activations in between, going from 100 inputs down to 10 outputs. At this point the layers exist but PyTorch has filled their weights with its *default* initialization — we will replace that next.

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(100, 100), nn.ReLU(),
    nn.Linear(100, 100), nn.ReLU(),
    nn.Linear(100, 10),
)

### Step 2 — Apply He initialization

Here is the actual initialization rule. For each `nn.Linear` layer we draw the weights from a **He (Kaiming) normal** distribution — the variance is scaled for ReLU so the signal neither shrinks to zero nor blows up as it passes through layers. Biases simply start at zero. `net.apply` walks every layer and runs this function on it.

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        # He (Kaiming) normal init, variance tuned for ReLU
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        # biases start at 0
        nn.init.zeros_(m.bias)

# walk every layer in the network and initialize it
net.apply(init_weights)

### Step 3 — Push a batch through and check the scale

The whole point of careful initialization is keeping activations at a sane scale. We send a random batch of 32 examples through the network and look at the standard deviation of the output. With He init it stays close to 1 — neither collapsed to 0 nor exploded.

In [ ]:
# a random input batch: 32 examples, 100 features each
x = torch.randn(32, 100)
out = net(x)

# with good init this stays sane — not 0, not huge
output_std = out.std().item()
print("output std:", round(output_std, 3))

## Visualize the data & results

_Pushing a real batch of digit images through 8 layers, does the activation size stay stable or explode?_

### Step 1 — Prepare a real digit batch

Instead of random noise we use a real batch of 512 handwritten-digit images. We scale the pixel values into [0, 1] and center them (subtract the per-feature mean), which is the kind of preprocessing a real network expects at its input.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()

# take 512 images and scale pixel intensities (0–16) into [0, 1]
batch = digits.data[:512] / 16.0
# center each feature by subtracting its mean
batch = batch - batch.mean(0)

rng = np.random.default_rng(3)
nh = 64   # hidden width of each layer we simulate

### Step 2 — Propagate through many layers at two scales

This helper pushes the batch through a stack of ReLU layers, recording the activation standard deviation after each one. The `scale` argument is the weight-init spread. We run it twice: once with the correct **He** scale `sqrt(2/nh)`, and once with a scale 2.5x too large.

In [ ]:
def propagate(scale, depth=8):
    # first layer maps the 64-dim centered batch into the hidden width
    first_weights = rng.standard_normal((64, nh)) * np.sqrt(2 / 64)
    x = np.maximum(0, batch @ first_weights)
    stds = [x.std()]
    for _ in range(depth):
        layer_weights = rng.standard_normal((nh, nh)) * scale
        x = np.maximum(0, x @ layer_weights)   # linear + ReLU
        stds.append(x.std())
    return stds

he = propagate(np.sqrt(2 / nh))          # correct He init scale
big = propagate(np.sqrt(2 / nh) * 2.5)   # scale 2.5x too large

### Step 3 — Plot how activation scale evolves with depth

On a log scale, the He curve stays roughly flat — the signal is preserved layer after layer. The too-large init grows exponentially with depth, which in a real network would saturate ReLUs and produce exploding gradients.

In [ ]:
plt.plot(he, color="#7ee787", label="He init (good)")
plt.plot(big, color="#ff7b72", label="too-large init (bad)")
plt.title("Activation std across layers on real digit batch (He vs too-large init)")
plt.xlabel("layer")
plt.ylabel("activation std (log scale)")
plt.yscale("log")
plt.legend()
plt.show()